# KTPM — benchmark Pha 1 trên Kaggle CPU
Import notebook, chọn **Accelerator: None**, bật **Internet**, thêm dataset chứa
`ktpm-p1-source.zip` do `scripts/export-kaggle.ps1` tạo. Không cần commit GitHub để chạy.
Backend, PostgreSQL thật và client tải cùng chạy trong phiên Kaggle này.
Chạy smoke trước, sau đó đổi `SMOKE = False` và chạy lại cell đo để lấy số liệu chính thức.
Đây là bộ chạy chuẩn bị sẵn; notebook chưa chứa kết quả đo được thực hiện trên Kaggle.
Xem `docs/KAGGLE_CPU.md` để hiểu giới hạn và so sánh Pha 2.

In [ ]:
import os, sys, json, hashlib, platform, subprocess, shutil, time, signal, zipfile
from pathlib import Path
from datetime import datetime, timezone
try:
    import psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'psutil==7.0.0'], check=True)
    import psutil

ROOT = Path('/kaggle/working/ktpm-benchmark')
ROOT.mkdir(parents=True, exist_ok=True)
matches = list(Path('/kaggle/input').rglob('ktpm-p1-source.zip'))
if len(matches) != 1:
    raise RuntimeError('Cần đúng một ktpm-p1-source.zip trong input dataset; chỉnh SOURCE_ZIP nếu có nhiều bản.')
SOURCE_ZIP = matches[0]
SOURCE = ROOT / 'source-p1'
SOURCE.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for entry in archive.infolist():
        destination = (SOURCE / entry.filename).resolve()
        if not destination.is_relative_to(SOURCE.resolve()) or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
            raise RuntimeError('Archive chứa đường dẫn không hợp lệ')
    archive.extractall(SOURCE)
BACKEND = SOURCE / 'backend'
RUNNER = SOURCE / 'scripts/benchmark.py'
RUNNER_HASH = hashlib.sha256(RUNNER.read_bytes()).hexdigest()
SMOKE = True
CONCURRENCIES = [1, 3] if SMOKE else [1, 10, 50]
REQUESTS = 30 if SMOKE else 1000
REPEATS = 1 if SMOKE else 3
MODES = ['read', 'bid', 'mixed']
print('Source hash:', hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest())
print('Runner hash:', RUNNER_HASH)
print('SMOKE:', SMOKE, 'requests:', REQUESTS, 'concurrency:', CONCURRENCIES)


## Chuẩn bị Java, dependency và tài khoản chạy PostgreSQL
PostgreSQL không chạy bằng root. Cell tạo user `ktpmbench` khi kernel đang là root.
Maven cache ở thư mục benchmark, có quyền đọc cho user này. Lần đầu cần Internet.
Build chỉ diễn ra trước đo; không tính thời gian tải/build vào kết quả.

In [ ]:
if shutil.which('java') is None:
    if os.geteuid() != 0:
        raise RuntimeError('Kernel thiếu Java; cần cài JDK 17+ trước khi chạy.')
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'], check=True)
if os.geteuid() == 0:
    if shutil.which('runuser') is None:
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'util-linux'], check=True)
    if subprocess.run(['id', 'ktpmbench'], capture_output=True).returncode:
        subprocess.run(['useradd', '--system', '--create-home', 'ktpmbench'], check=True)
    AS_USER = ['runuser', '-u', 'ktpmbench', '--']
else:
    AS_USER = []
M2 = ROOT / 'm2'
CP_FILE = ROOT / 'classpath.txt'
subprocess.run(['bash', str(BACKEND / 'mvnw'), '-f', str(BACKEND / 'pom.xml'),
                f'-Dmaven.repo.local={M2}', 'test-compile',
                'org.apache.maven.plugins:maven-dependency-plugin:3.6.1:build-classpath',
                f'-Dmdep.outputFile={CP_FILE}'], check=True)
CLASSPATH = os.pathsep.join([str(BACKEND / 'target/test-classes'),
                            str(BACKEND / 'target/classes'), CP_FILE.read_text().strip()])
OUT = ROOT / ('results-smoke' if SMOKE else 'results-p1')
OUT.mkdir(exist_ok=True)
environment = {
    'recordedAtUtc': datetime.now(timezone.utc).isoformat(),
    'sourceSha256': hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest(),
    'runnerSha256': RUNNER_HASH, 'platform': platform.platform(),
    'python': sys.version, 'logicalCpus': psutil.cpu_count(),
    'affinity': psutil.Process().cpu_affinity(), 'memoryBytes': psutil.virtual_memory().total,
    'java': subprocess.run(['java', '-version'], capture_output=True, text=True).stderr,
    'cpu': subprocess.run(['lscpu'], capture_output=True, text=True).stdout,
    'psutil': psutil.__version__, 'smoke': SMOKE,
    'requests': REQUESTS, 'concurrencies': CONCURRENCIES, 'repeats': REPEATS,
    'warmupRequests': 100, 'users': 25, 'seed': 42,
    'jvmOptions': ['-Xms256m', '-Xmx512m', '-XX:ActiveProcessorCount=2'],
    'postgresVersion': '16.2 (embedded binaries BOM)',
    'topology': 'one backend + PostgreSQL + load generator on same Kaggle CPU kernel',
    'limits': 'Closed-loop client; co-located client consumes CPU. Not an Internet latency or capacity SLA test.'
}
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
print(environment['java'], 'Logical CPUs:', environment['logicalCpus'])


## Hàm chạy từng lần đo
Mỗi trial tạo database mới trong **runtime của notebook**, 25 bidder, một phiên ACTIVE,
warmup 100 request, rồi đo. Không sử dụng database trên máy Windows.
CPU/RSS chỉ lấy cây tiến trình Java và PostgreSQL; client vẫn cạnh tranh cùng CPU.
Fixture chứa token được xóa sau trial; không đưa vào file kết quả tải xuống.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('ktpm_benchmark', RUNNER)
bench = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bench)
from types import SimpleNamespace

def stop_server(process):
    if process is None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline:
        process.poll()
        try:
            os.killpg(process.pid, 0)
        except ProcessLookupError:
            return
        time.sleep(.25)
    # Only this notebook's process group, never all Java/PostgreSQL processes.
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait(timeout=10)

def trial(mode, concurrency, repeat):
    label = f'{mode}-c{concurrency}-r{repeat}'
    runtime = ROOT / ('runtime-' + label)
    runtime.mkdir(exist_ok=True)
    pg_data = runtime / 'postgres'
    if pg_data.exists():
        # Refuse to erase a database whose prior server may still own it.
        pid_file = pg_data / 'postmaster.pid'
        if pid_file.exists():
            previous_pid = int(pid_file.read_text().splitlines()[0])
            if psutil.pid_exists(previous_pid):
                raise RuntimeError('Trial cũ vẫn chạy; dừng kernel/process trước khi chạy lại.')
        assert pg_data.resolve().parent == runtime.resolve()
        shutil.rmtree(pg_data)
    if AS_USER:
        subprocess.run(['chown', '-R', 'ktpmbench:ktpmbench', str(runtime)], check=True)
    fixture = runtime / 'fixture.json'
    log = (runtime / 'backend.log').open('w')
    process = None
    try:
        env = os.environ.copy()
        import secrets
        env['JWT_SECRET'] = secrets.token_urlsafe(48)
        command = AS_USER + ['java', '-Xms256m', '-Xmx512m', '-XX:ActiveProcessorCount=2',
                             f'-Dktpm.local.data={pg_data}', '-cp', CLASSPATH, 'com.ktpm.LocalDemo']
        process = subprocess.Popen(command, cwd=BACKEND, env=env, stdout=log,
                                   stderr=subprocess.STDOUT, start_new_session=True)
        deadline = time.monotonic() + 120
        while bench.api('http://localhost:8080', '/actuator/health', timeout=2)[0] != 200:
            if process.poll() is not None or time.monotonic() > deadline:
                raise RuntimeError(f'Backend chưa sẵn sàng; xem {runtime}/backend.log')
            time.sleep(1)
        candidates = [psutil.Process(process.pid)] + psutil.Process(process.pid).children(recursive=True)
        java_pid = next(p.pid for p in candidates if p.name().startswith('java'))
        database_pid = int((pg_data / 'postmaster.pid').read_text().splitlines()[0])
        bench.prepare(SimpleNamespace(base='http://localhost:8080', users=25, fixture=str(fixture)))
        common = dict(fixture=str(fixture), mode=mode, seed=42, timeout=30,
                      server_pid=java_pid, database_pid=database_pid)
        bench.run(SimpleNamespace(**common, requests=100, concurrency=1, phase='warmup',
                                  output=str(runtime / 'warmup.json'), samples_output=None))
        return bench.run(SimpleNamespace(**common, requests=REQUESTS, concurrency=concurrency,
                                        phase='P1', output=str(OUT / (label + '.json')),
                                        samples_output=str(OUT / (label + '.jsonl'))))
    finally:
        stop_server(process)
        log.close()
        fixture.unlink(missing_ok=True)


## Đo và xuất kết quả
Chạy cell này hoàn tất mới tải kết quả. Khi đổi SMOKE phải chạy lại cell cấu hình và
cell môi trường trước cell này. 409 là xung đột bid hợp lệ, cần trình bày riêng;
RPS thành công khác tổng RPS. Không coi request bị từ chối là bid thành công.

In [ ]:
reports = []
for mode in MODES:
    for concurrency in CONCURRENCIES:
        for repeat in range(1, REPEATS + 1):
            print('Trial:', mode, concurrency, repeat)
            reports.append(trial(mode, concurrency, repeat))

import csv, statistics
rows = []
for mode in MODES:
    for concurrency in CONCURRENCIES:
        group = [r for r in reports if r['mode'] == mode and r['concurrency'] == concurrency]
        successful_p95 = [r['latencySuccessful']['p95Ms'] for r in group
                          if r['latencySuccessful']['p95Ms'] is not None]
        rows.append(dict(mode=mode, concurrency=concurrency, repeats=len(group),
                         medianTotalRps=statistics.median(r['throughputRequestsPerSecond'] for r in group),
                         medianSuccessfulRps=statistics.median(r['successfulRequestsPerSecond'] for r in group),
                         medianTrialP95Ms=statistics.median(r['latencyAll']['p95Ms'] for r in group),
                         medianSuccessfulTrialP95Ms=statistics.median(successful_p95) if successful_p95 else None,
                         medianConflictRate=statistics.median(r['businessConflictRate'] for r in group),
                         medianUnexpectedErrorRate=statistics.median(r['unexpectedErrorRate'] for r in group)))
with (OUT / 'summary.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(*rows, sep='\n')
archive_path = ROOT / (OUT.name + '.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(OUT.iterdir()):
        if file.suffix in ('.json', '.jsonl', '.csv'):
            archive.write(file, file.name)
print('Download:', archive_path)


## So sánh Pha 2
Giữ source ZIP P1 và runner SHA256 này làm baseline. Với P2, dùng cùng notebook,
thay SOURCE_ZIP/SOURCE/BACKEND sang ZIP mới, đổi nhãn `phase`/thư mục OUT thành P2,
nhưng **giữ nguyên RUNNER P1**, JVM, PostgreSQL, seed, dữ liệu, warmup và ma trận tải.
Nên đo P1 và P2 trong cùng phiên kernel, kiểm tra `lscpu`, affinity, RAM và phiên bản
trong environment.json. Nếu kernel đổi cấu hình, đo lại cả hai; không ghép số đo máy
Windows với Kaggle. Nếu cải tiến cần nhiều instance, ghi rõ topology và giữ tổng tài nguyên.
So sánh median từng nhóm qua 3 lần, p95/p99, successful RPS, 409, lỗi ngoài dự kiến,
CPU/RSS; đọc raw JSONL khi phân tích biến động. Median p95 của các trial không phải
p95 gộp. Không kết luận bottleneck DB chỉ từ CPU tổng: cần thêm truy vấn/lock metrics ở P2.